<a href="https://colab.research.google.com/github/ipavlopoulos/ndfu/blob/main/ndfu_mechanism.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# How nDFU works internally

nDFU scores whether an ordinal annotation histogram behaves like a single-peaked distribution. The idea is simple: once we start from the highest bin and move away from it, the frequencies should not rise again. If they do rise again, that upward jump is evidence of another pole.

This notebook walks through the exact mechanism used by the package implementation.

## The algorithm

Given a histogram `hist`:

1. Find the maximum histogram value.
2. Choose the first bin where that maximum occurs. This is the implementation's peak.
3. Move right from the peak. A unimodal histogram should stay flat or decrease. Any positive value of `hist[i + 1] - hist[i]` is a violation.
4. Move left from the peak. A unimodal histogram should stay flat or decrease. Any positive value of `hist[i - 1] - hist[i]` is a violation.
5. The raw DFU score is the largest positive violation.
6. nDFU divides that raw score by the peak height.

So, informally:

`nDFU = largest upward jump away from the peak / peak height`

A score of `0` means there is no upward jump away from the peak. A larger score means the histogram rises again somewhere away from the peak, suggesting separated poles.

In [ ]:
#@title Setup
try:
    from ndfu import dfu, pdf
except ImportError:
    import sys
    import subprocess

    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "git+https://github.com/ipavlopoulos/ndfu.git",
    ])
    from ndfu import dfu, pdf

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

sns.set_theme(style="whitegrid", context="notebook")
SCALE = list(range(1, 6))


## A transparent implementation

The helper below mirrors `ndfu.dfu`, but it also returns the intermediate comparisons so we can inspect the score.

In [ ]:
#@title Explain one nDFU calculation
def explain_dfu(hist):
    hist = np.asarray(hist, dtype=float)
    peak_height = hist.max()
    peak_pos = int(np.flatnonzero(hist == peak_height)[0])

    rows = []
    max_violation = 0.0

    for i in range(peak_pos, len(hist) - 1):
        violation = hist[i + 1] - hist[i]
        max_violation = max(max_violation, violation)
        rows.append({
            "side": "right",
            "from_rating": SCALE[i],
            "to_rating": SCALE[i + 1],
            "from_frequency": hist[i],
            "to_frequency": hist[i + 1],
            "upward_jump": max(0.0, violation),
        })

    for i in range(peak_pos, 0, -1):
        violation = hist[i - 1] - hist[i]
        max_violation = max(max_violation, violation)
        rows.append({
            "side": "left",
            "from_rating": SCALE[i],
            "to_rating": SCALE[i - 1],
            "from_frequency": hist[i],
            "to_frequency": hist[i - 1],
            "upward_jump": max(0.0, violation),
        })

    raw_dfu = max_violation
    ndfu = raw_dfu / peak_height
    table = pd.DataFrame(rows)
    return peak_pos, peak_height, table, raw_dfu, ndfu


def plot_hist(hist, title, peak_pos=None):
    colors = ["#cbd5e1"] * len(hist)
    if peak_pos is not None:
        colors[peak_pos] = "#0f766e"

    fig, ax = plt.subplots(figsize=(8, 2.8))
    pd.Series(hist, index=SCALE).plot.bar(ax=ax, color=colors, width=0.8)
    ax.set_title(title)
    ax.set_xlabel("Ordinal rating")
    ax.set_ylabel("Relative frequency")
    ax.set_ylim(0, max(0.6, float(np.max(hist)) + 0.1))
    ax.bar_label(ax.containers[0], fmt="%.2f", padding=2, fontsize=9)
    sns.despine(ax=ax, left=True, bottom=True)
    plt.show()
    plt.close(fig)


## Example 1: two clean poles

Half the annotators choose `1` and half choose `5`. The first peak is rating `1`. Moving right from that peak, the histogram eventually rises from `0.00` to `0.50` at rating `5`. That upward jump is the second pole.

In [ ]:
#@title Clean poles: ratings 1 and 5
ratings = [1] * 12 + [5] * 12
hist = pdf(ratings, SCALE)
peak_pos, peak_height, table, raw_dfu, score = explain_dfu(hist)

plot_hist(hist, "Two clean poles", peak_pos)
print(f"Peak rating: {SCALE[peak_pos]} with height {peak_height:.2f}")
print(f"Raw DFU: {raw_dfu:.2f}")
print(f"nDFU: {score:.2f}")
display(table)
assert score == dfu(hist)


## Example 2: one peak

A unimodal histogram never rises as we move away from the peak. Because every outward comparison is flat or decreasing, the largest upward jump is `0`, and nDFU is `0`.

In [ ]:
#@title Unimodal ratings
ratings = [1] * 2 + [2] * 4 + [3] * 12 + [4] * 4 + [5] * 2
hist = pdf(ratings, SCALE)
peak_pos, peak_height, table, raw_dfu, score = explain_dfu(hist)

plot_hist(hist, "One shared peak", peak_pos)
print(f"Peak rating: {SCALE[peak_pos]} with height {peak_height:.2f}")
print(f"Raw DFU: {raw_dfu:.2f}")
print(f"nDFU: {score:.2f}")
display(table)
assert score == dfu(hist)


## Example 3: poles softened by middle ratings

If some annotators choose middle ratings, the valley is no longer empty. The largest upward jump away from the first peak becomes smaller, so nDFU decreases while still indicating structured disagreement.

In [ ]:
#@title Poles with some middle ratings
ratings = [1] * 10 + [2, 3, 3, 4] + [5] * 10
hist = pdf(ratings, SCALE)
peak_pos, peak_height, table, raw_dfu, score = explain_dfu(hist)

plot_hist(hist, "Poles with a shallower valley", peak_pos)
print(f"Peak rating: {SCALE[peak_pos]} with height {peak_height:.2f}")
print(f"Raw DFU: {raw_dfu:.2f}")
print(f"nDFU: {score:.2f}")
display(table)
assert score == dfu(hist)


## Why normalize?

Raw DFU is measured in histogram-frequency units. Dividing by the peak height makes the score relative to the scale of the histogram. This is why `[0.5, 0.0, 0.5]` has raw DFU `0.5` but normalized DFU `1.0`: the upward jump is as large as the peak itself.

In [ ]:
#@title Raw DFU vs normalized DFU
hist = np.array([0.5, 0.0, 0.5])
peak_pos, peak_height, table, raw_dfu, score = explain_dfu(hist)

pd.DataFrame([
    {"quantity": "peak height", "value": peak_height},
    {"quantity": "largest upward jump", "value": raw_dfu},
    {"quantity": "largest upward jump / peak height", "value": score},
])


## Takeaway

Internally, nDFU is a peak-centered monotonicity check. A unimodal annotation histogram should not rise again as we move away from its peak. nDFU measures the strongest rise away from the selected peak, normalized by the peak height. That is what makes it useful for detecting pole-like annotator disagreement.